# Bond Option Comparison with DerivaGem Function 17

Value an option on a coupon paying bond using the Hull-White in response to a question.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time

In [2]:
from financepy.utils import *
from financepy.market.curves.flat_discount_curve import FlatDiscountCurve
from financepy.models.hw_tree import HWTree
from financepy.products.bonds.bond_option import *

#############################################################
#  FINANCEPY Version 1.1.2 - Built on 25 Sep 2026 at 14:07  #
#  This software is distributed FREE AND WITHOUT WARRANTY   #
#  Report issues at https://github.com/domokane/FinancePy   #
#############################################################



## Set up the Discount Curve

In [3]:
settle_dt = Date(1, 12, 2019)

Set up discount curve

In [4]:
rate = 0.05
dc_type = DayCountTypes.THIRTY_360_BOND
fixed_freq = FrequencyTypes.SEMI_ANNUAL
discount_curve = FlatDiscountCurve(settle_dt, rate, fixed_freq, dc_type)

In [5]:
df_times = discount_curve._times
df_values = discount_curve._dfs

In [6]:
df_values

array([1.        , 0.9518144 , 0.90595064, 0.86229687, 0.82074657,
       0.7811984 ])

## Set up the Bond Option

First create the bond

In [7]:
issue_dt = Date(1,12,2018)
maturity_dt = issue_dt.add_tenor("10Y")
coupon = 0.05
freq_type = FrequencyTypes.SEMI_ANNUAL
dc_type = DayCountTypes.THIRTY_360_BOND
bond = Bond(issue_dt, maturity_dt, coupon, freq_type, dc_type)

Let's first price the bond on the libor curve

In [8]:
cp = bond.clean_price_from_discount_curve(settle_dt, discount_curve)
dp = bond.dirty_price_from_discount_curve(settle_dt, discount_curve)
print("Fixed Income Clean Price: %9.3f"% cp)
print("Fixed Income Dirty Price: %9.3f"% dp)

Fixed Income Clean Price:    99.999
Fixed Income Dirty Price:    99.999


In [9]:
settle_dt

01-DEC-2019

In [10]:
bond.print_payments(settle_dt)

Coupon Date 	 Payment Date 	         Status          Amount
01-DEC-2019 	             	     SETTLEMENT 
01-JUN-2020 	 01-JUN-2020 	      UNCHANGED 	            2.50 
01-DEC-2020 	 01-DEC-2020 	      UNCHANGED 	            2.50 
01-JUN-2021 	 01-JUN-2021 	      UNCHANGED 	            2.50 
01-DEC-2021 	 01-DEC-2021 	      UNCHANGED 	            2.50 
01-JUN-2022 	 01-JUN-2022 	      UNCHANGED 	            2.50 
01-DEC-2022 	 01-DEC-2022 	      UNCHANGED 	            2.50 
01-JUN-2023 	 01-JUN-2023 	      UNCHANGED 	            2.50 
01-DEC-2023 	 01-DEC-2023 	      UNCHANGED 	            2.50 
01-JUN-2024 	 03-JUN-2024 	   HOLIDAY ROLL 	            2.50 
01-DEC-2024 	 02-DEC-2024 	   HOLIDAY ROLL 	            2.50 
01-JUN-2025 	 02-JUN-2025 	   HOLIDAY ROLL 	            2.50 
01-DEC-2025 	 01-DEC-2025 	      UNCHANGED 	            2.50 
01-JUN-2026 	 01-JUN-2026 	      UNCHANGED 	            2.50 
01-DEC-2026 	 01-DEC-2026 	      UNCHANGED 	            2.50 
01-JUN-2027 	 01-JUN-2027 	 

Then define and create the option

In [11]:
expiry_dt = settle_dt.add_tenor("18m")
strike_price = 100.0
face_amount = 100.0

In [12]:
europeanCallBondOption = BondOption(bond, expiry_dt, strike_price, OptionTypes.EUROPEAN_CALL)
americanCallBondOption = BondOption(bond, expiry_dt, strike_price, OptionTypes.AMERICAN_CALL)

In [13]:
europeanPutBondOption = BondOption(bond, expiry_dt, strike_price, OptionTypes.EUROPEAN_PUT)
americanPutBondOption = BondOption(bond, expiry_dt, strike_price, OptionTypes.AMERICAN_PUT)

Consider the forward bond price.

In [14]:
cp = bond.clean_price_from_discount_curve(expiry_dt, discount_curve)
dp = bond.dirty_price_from_discount_curve(expiry_dt, discount_curve)
print("Fixed Income Clean Price: %9.3f"% cp)
print("Fixed Income Dirty Price: %9.3f"% dp)

Fixed Income Clean Price:    99.999
Fixed Income Dirty Price:    99.999


Set the model parameters, start with zero vol.

In [15]:
num_steps = 500

In [16]:
sigma = 0.0125
a = 0.1
modelHW = HWTree(sigma, a, num_steps)

In [17]:
ec = europeanCallBondOption.value(settle_dt, discount_curve, modelHW)
ac = americanCallBondOption.value(settle_dt, discount_curve, modelHW)
print("European Call Value: %9.5f" % ec)
print("American Call Value: %9.5f" % ac)

European Call Value:   2.39721
American Call Value:   2.66567


In [18]:
ec = europeanPutBondOption.value(settle_dt, discount_curve, modelHW)
ac = americanPutBondOption.value(settle_dt, discount_curve, modelHW)
print("European Put Value: %9.5f" % ec)
print("American Put Value: %9.5f" % ac)

European Put Value:   2.42317
American Put Value:   2.68335


# COMPARISON

In [19]:
if 1==1:

    settle_dt = Date(1, 12, 2019)
    issue_dt = Date(1, 12, 2018)
    expiry_dt = settle_dt.add_tenor("18m")
    maturity_dt = settle_dt.add_tenor("10Y")
    coupon = 0.05
    freq_type = FrequencyTypes.SEMI_ANNUAL
    accrual_type = DayCountTypes.THIRTY_360_BOND
    bond = Bond(issue_dt, maturity_dt, coupon, freq_type,  accrual_type)

    couponTimes = []
    couponFlows = []
    cpn = bond.cpn/bond.freq

    numFlows = len(bond.cpn_dts)
    for i in range(0, numFlows):

        pcd = bond.cpn_dts[i-1]
        ncd = bond.cpn_dts[i]

        if ncd > settle_dt:

            if len(couponTimes) == 0:
                flowTime = (pcd - settle_dt) / G_DAYS_IN_YEAR
                couponTimes.append(flowTime)
                couponFlows.append(cpn)

            flowTime = (ncd - settle_dt) / G_DAYS_IN_YEAR
            couponTimes.append(flowTime)
            couponFlows.append(cpn)

    couponTimes = np.array(couponTimes)
    couponFlows = np.array(couponFlows)

    strikePrice = 100.0
    face = 100.0

    model = HWTree(sigma, a, None)

    #  Test convergence
    t_exp = (expiry_dt - settle_dt)/G_DAYS_IN_YEAR
    t_mat = (maturity_dt - settle_dt)/G_DAYS_IN_YEAR

    # Jamshidian approach
    v_jam = model.european_bond_option_jamshidian(t_exp, strikePrice, face,
                                              couponTimes, couponFlows,
                                              df_times, df_values)

    model.num_time_steps = 200
    model.build_tree(t_mat, df_times, df_values)
    exerciseType = ExerciseTypes.EUROPEAN
    vHW = model.bond_option(t_exp, strikePrice, face,
                            couponTimes, couponFlows, exerciseType)


In [20]:
v_jam

(np.float64(2.5571679355944985), np.float64(2.582591550453939))

In [21]:
vHW

(2.5651092396174104, 2.5882400815731788)

Copyright (c) Dominic O'Kane 2021